In [2]:
import pandas as pd

In [3]:
rnamigo2_robin_df = pd.read_csv('./display_outputs/rnamigos2_robin_tpp_results.csv')

In [ ]:
# get the max number in column dock and print the smiles
dock_values = rnamigo2_robin_df['dock']
print(max(dock_values))

# get the smiles of the max number
max_smiles = rnamigo2_robin_df[rnamigo2_robin_df['dock'] == max(dock_values)]['smiles']
print(max_smiles)
#

0.21273175
24082    [H][C@@]12C(N[C@@]3([H])C(N[C@@]4([H])C(N[C@](...
Name: smiles, dtype: object


## Optimizer Comparison Results

Analysis of 4 different molecular optimization methods across 5 random seeds:
- **Graph GA**: Graph-based genetic algorithm
- **SMILES GA**: SMILES-based genetic algorithm  
- **REINVENT**: Deep learning-based generative model
- **Screening**: Random baseline (no optimization)

Each optimizer ran for 1500 oracle evaluations on the RNAmigos2 oracle targeting TPP riboswitch (2gdi).

In [5]:
import yaml
import numpy as np
from pathlib import Path

# Define the optimizers and their metrics files
optimizers = ['graph_ga', 'smiles_ga', 'reinvent', 'gp_bo']
seeds = [0] # , 1, 2, 3, 5
results_dir = Path("opt_results") / "run_combined_sim"
oracle_name = "oracle"

# Read all metrics
all_metrics = {}
for optimizer in optimizers:
    optimizer_data = []
    for seed in seeds:
        file_path = f'{results_dir}/{optimizer}/metrics_{optimizer}_{oracle_name}_{seed}.yaml'
        with open(file_path, 'r') as f:
            metrics = yaml.safe_load(f)
            optimizer_data.append(metrics)
    all_metrics[optimizer] = optimizer_data

print("Successfully loaded metrics for all optimizers!")
print(f"Metrics per run: {list(all_metrics['graph_ga'][0].keys())}")

Successfully loaded metrics for all optimizers!
Metrics per run: ['avg_top1', 'avg_top10', 'avg_top100', 'auc_top1', 'auc_top10', 'auc_top100', 'avg_sa', 'diversity_top100', 'n_oracle']


In [6]:
# Calculate mean and std for each metric (excluding n_oracle)
metric_names = [k for k in all_metrics['graph_ga'][0].keys() if k != 'n_oracle']

results = {}
for optimizer in optimizers:
    optimizer_results = {}
    for metric in metric_names:
        values = [run[metric] for run in all_metrics[optimizer]]
        mean = np.mean(values)
        std = np.std(values, ddof=1)  # Use sample std (n-1)
        optimizer_results[metric] = f"{mean:.3f} ± {std:.3f}"
    results[optimizer] = optimizer_results

# Create DataFrame with optimizers as columns and metrics as rows
results_df = pd.DataFrame(results)
results_df = results_df.T  # Transpose so optimizers are rows
results_df.index.name = 'Optimizer'

print("Metrics Summary (Mean ± Std across 5 seeds):")
print("=" * 80)
results_df

Metrics Summary (Mean ± Std across 5 seeds):


/home/wangx86/miniconda3/envs/rnamigo2/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/wangx86/miniconda3/envs/rnamigo2/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,avg_top1,avg_top10,avg_top100,auc_top1,auc_top10,auc_top100,avg_sa,diversity_top100
Optimizer,,,,,,,,
graph_ga,0.393 ± nan,0.328 ± nan,0.259 ± nan,0.291 ± nan,0.242 ± nan,0.170 ± nan,3.555 ± nan,0.797 ± nan
smiles_ga,0.602 ± nan,0.463 ± nan,0.304 ± nan,0.278 ± nan,0.220 ± nan,0.132 ± nan,6.225 ± nan,0.810 ± nan
reinvent,2.915 ± nan,2.912 ± nan,2.905 ± nan,1.585 ± nan,1.529 ± nan,1.437 ± nan,8.719 ± nan,0.132 ± nan
gp_bo,1.341 ± nan,1.122 ± nan,0.908 ± nan,0.715 ± nan,0.623 ± nan,0.493 ± nan,5.844 ± nan,0.686 ± nan


In [8]:
# Create the final table with metrics as rows and optimizers as columns
final_results = {}
for optimizer in optimizers:
    optimizer_stats = {}
    for metric in metric_names:
        values = [run[metric] for run in all_metrics[optimizer]]
        mean = np.mean(values)
        std = np.std(values, ddof=1)
        optimizer_stats[metric] = f"{mean:.3f} ± {std:.3f}"
    final_results[optimizer] = optimizer_stats

# Create DataFrame with metrics as rows and optimizers as columns
metrics_table = pd.DataFrame(final_results)
metrics_table.index.name = 'Metric'

# Rename columns for better readability
metrics_table.columns = ['Graph GA', 'SMILES GA', 'REINVENT', 'GP BO']

print("\n" + "="*100)
print("OPTIMIZATION RESULTS: Mean ± Std across 5 random seeds (0, 1, 2, 3, 5)")
print("="*100)
print(metrics_table)
print("\n")


OPTIMIZATION RESULTS: Mean ± Std across 5 random seeds (0, 1, 2, 3, 5)
                     Graph GA    SMILES GA     REINVENT        GP BO
Metric                                                              
avg_top1          0.393 ± nan  0.602 ± nan  2.915 ± nan  1.341 ± nan
avg_top10         0.328 ± nan  0.463 ± nan  2.912 ± nan  1.122 ± nan
avg_top100        0.259 ± nan  0.304 ± nan  2.905 ± nan  0.908 ± nan
auc_top1          0.291 ± nan  0.278 ± nan  1.585 ± nan  0.715 ± nan
auc_top10         0.242 ± nan  0.220 ± nan  1.529 ± nan  0.623 ± nan
auc_top100        0.170 ± nan  0.132 ± nan  1.437 ± nan  0.493 ± nan
avg_sa            3.555 ± nan  6.225 ± nan  8.719 ± nan  5.844 ± nan
diversity_top100  0.797 ± nan  0.810 ± nan  0.132 ± nan  0.686 ± nan




/home/wangx86/miniconda3/envs/rnamigo2/lib/python3.10/site-packages/numpy/core/_methods.py:206: RuntimeWarning: Degrees of freedom <= 0 for slice
  ret = _var(a, axis=axis, dtype=dtype, out=out, ddof=ddof,
/home/wangx86/miniconda3/envs/rnamigo2/lib/python3.10/site-packages/numpy/core/_methods.py:198: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


In [10]:
# Display the styled table
metrics_table

,Graph GA,SMILES GA,REINVENT,GP BO
Metric,,,,
avg_top1,0.393 ± nan,0.602 ± nan,2.915 ± nan,1.341 ± nan
avg_top10,0.328 ± nan,0.463 ± nan,2.912 ± nan,1.122 ± nan
avg_top100,0.259 ± nan,0.304 ± nan,2.905 ± nan,0.908 ± nan
auc_top1,0.291 ± nan,0.278 ± nan,1.585 ± nan,0.715 ± nan
auc_top10,0.242 ± nan,0.220 ± nan,1.529 ± nan,0.623 ± nan
auc_top100,0.170 ± nan,0.132 ± nan,1.437 ± nan,0.493 ± nan
avg_sa,3.555 ± nan,6.225 ± nan,8.719 ± nan,5.844 ± nan
diversity_top100,0.797 ± nan,0.810 ± nan,0.132 ± nan,0.686 ± nan


In [12]:
# save the metric table
metrics_table.to_csv("display_outputs/combined_sim_optimizers_metrics.csv")

### Summary

The **metrics table** (`metrics_table`) contains mean ± std for each metric across 5 random seeds.

**Key Metrics Explained:**
- `avg_top1/10/100`: Average RNAmigos2 score of the top 1/10/100 molecules found
- `auc_top1/10/100`: Area under the curve for top scoring molecules over optimization time
- `avg_sa`: Average Synthetic Accessibility score (2-10 scale, lower = easier to synthesize)
- `diversity_top100`: Tanimoto diversity among top 100 molecules (0-1, higher = more diverse)